# Preparación del dataset semanal

Construye el dataset semanal agrupando retornos logarítmicos diarios. Las features incluyen el retorno de cada ETF, la variación de volumen y variables de calendario. Se generan cinco versiones: sin lags y con 1, 2, 3 y 4 semanas de histórico adicional.

In [ ]:
import pandas as pd
import numpy as np

# Datos del preprocesado
adj_close = pd.read_csv(
    "../Datos_csv/adj_close.csv",
    index_col=0,
    parse_dates=True
)

volume = pd.read_csv(
    "../Datos_csv/volume.csv",
    index_col=0,
    parse_dates=True
)

In [310]:
to_drop = ['BIL', 'EWY', 'GLD', 'IAU', 'SHY']

adj_close = adj_close.drop(columns=to_drop)
volume    = volume.drop(columns=to_drop)


In [311]:
# Retornos logarítmicos diarios
ret_log_diff = np.log(adj_close).diff()

In [312]:
ret_log_diff=ret_log_diff[1:]

In [313]:
# también recortamos volumen para que empiece en las mismas fechas
volume = volume.loc[ret_log_diff.index].copy()


In [314]:

fechas = ret_log_diff.index.to_series()
iso_cal = ret_log_diff.index.isocalendar()
week_info = pd.DataFrame(index=ret_log_diff.index)

# Año ISO y número de semana ISO
week_info["year"] = iso_cal.year.astype(int)
week_info["semana"] = iso_cal.week.astype(int)

# Mes natural de cada fecha
week_info["mes"] = fechas.dt.month.values

# Claves de mes y trimestre para detectar cambios reales en días de trading
periodo_mes = fechas.dt.to_period("M")
periodo_trimestre = fechas.dt.to_period("Q")

# INICIO DE MES REAL:
# 1 si esa fecha es el primer día de trading de su mes, Esto compara cada fecha con la anterior:
#si el mes cambia respecto al registro anterior, entonces esa fecha es el primer día de trading del nuevo mes
week_info["inicio_mes"] = (periodo_mes != periodo_mes.shift(1)).astype(int).values

# FIN DE MES REAL:
# 1 si esa fecha es el último día de trading de su mes
week_info["fin_mes"] = (periodo_mes != periodo_mes.shift(-1)).astype(int).values

# FIN DE TRIMESTRE REAL:
# 1 si esa fecha es el último día de trading de su trimestre
week_info["fin_trimestre"] = (periodo_trimestre != periodo_trimestre.shift(-1)).astype(int).values
#inicio trimestre
week_info["inicio_trimestre"] = (periodo_trimestre != periodo_trimestre.shift(1)).astype(int).values

# Clave semana-año para no mezclar semanas de años distintos
week_info["week_key"] = (
    week_info["year"].astype(str)
    + "-W"
    + week_info["semana"].astype(str).str.zfill(2)
)

Se usa el primer y último día de trading real del mes/trimestre, no el día natural, para evitar que caiga en fin de semana o festivo.

In [315]:
# Retorno semanal por ETF = suma de retornos log diarios
ret_weekly = ret_log_diff.groupby(week_info["week_key"]).mean() 
ret_weekly = ret_weekly.add_prefix("ret_semanal_") 

# Volumen semanal por ETF = suma semanal
vol_weekly = volume.groupby(week_info["week_key"]).mean()
vol_weekly = vol_weekly.add_prefix("volumen_semanal_")

# Variables temporales semanales
year_weekly = week_info.groupby("week_key")["year"].first()
semana_weekly = week_info.groupby("week_key")["semana"].first()
mes_weekly = week_info.groupby("week_key")["mes"].last()

# Si la semana contiene al menos un día que sea:
# - primer día de trading del mes
# - último día de trading del mes
# - último día de trading del trimestre
inicio_mes_weekly = week_info.groupby("week_key")["inicio_mes"].max()
fin_mes_weekly = week_info.groupby("week_key")["fin_mes"].max()
inicio_trimestre_weekly = week_info.groupby("week_key")["inicio_trimestre"].max()
fin_trimestre_weekly = week_info.groupby("week_key")["fin_trimestre"].max()

# DataFrame base semanal
df_semanal = pd.DataFrame(index=ret_weekly.index)
df_semanal["year"] = year_weekly
df_semanal["semana"] = semana_weekly
df_semanal["mes"] = mes_weekly
df_semanal["inicio_mes"] = inicio_mes_weekly
df_semanal["fin_mes"] = fin_mes_weekly
df_semanal["inicio_trimestre"] = inicio_trimestre_weekly
df_semanal["fin_trimestre"] = fin_trimestre_weekly

# Unir todo
df_semanal = pd.concat(
    # [df_semanal, ret_weekly, vol_weekly, n_dias_weekly],
    [df_semanal, ret_weekly, vol_weekly],
    axis=1
).reset_index().rename(columns={"index": "week_key"})

Diferencias logarítmicas del volumen:

Mide la variación relativa del volumen entre semanas, para detectar aumentos o caídas en la actividad del mercado.

Ratio de volumen: compara el volumen actual con la media de las últimas 4 semanas para detectar actividad anormal.

In [316]:
etfs = adj_close.columns.tolist()

vol_features = {}

for etf in etfs:
    col_vol = f"volumen_semanal_{etf}"

    # cambio logarítmico del volumen
    vol_features[f"volumen_log_diff_{etf}"] = (
        np.log(df_semanal[col_vol].replace(0, np.nan)).diff()
    )

# pasar todo a DataFrame de una vez
vol_features_df = pd.DataFrame(vol_features, index=df_semanal.index)

# unir de una vez y desfragmentar
df_semanal = pd.concat([df_semanal, vol_features_df], axis=1).copy()

# eliminar volumen absoluto
cols_vol = [f"volumen_semanal_{etf}" for etf in etfs]
df_semanal = df_semanal.drop(columns=cols_vol).copy()


In [317]:

# 9. TARGETS: RETORNO DE LA SEMANA SIGUIENTE DE CADA ETF
targets_dict = {}

for etf in etfs:
    col_ret = f"ret_semanal_{etf}"
    targets_dict[f"target_{etf}"] = df_semanal[col_ret].shift(-1)

targets_df = pd.DataFrame(targets_dict, index=df_semanal.index)

df_semanal = pd.concat([df_semanal, targets_df], axis=1).copy()

In [318]:
import pandas as pd

pd.set_option("display.max_columns", None) 
df_semanal.head()

,week_key,year,semana,mes,inicio_mes,fin_mes,inicio_trimestre,fin_trimestre,ret_semanal_AGG,ret_semanal_BND,ret_semanal_DBC,ret_semanal_DIA,ret_semanal_DVY,ret_semanal_EEM,ret_semanal_EFA,ret_semanal_EWG,ret_semanal_EWJ,ret_semanal_EWQ,ret_semanal_EWT,ret_semanal_EWU,ret_semanal_EWZ,ret_semanal_FXI,ret_semanal_HYG,ret_semanal_IEF,ret_semanal_IEFA,ret_semanal_IEMG,ret_semanal_IJR,ret_semanal_INDA,ret_semanal_ITOT,ret_semanal_IVV,ret_semanal_IWD,ret_semanal_IWF,ret_semanal_IWM,ret_semanal_JNK,ret_semanal_LQD,ret_semanal_MCHI,ret_semanal_MDY,ret_semanal_MTUM,ret_semanal_QQQ,ret_semanal_QUAL,ret_semanal_SCHD,ret_semanal_SLV,ret_semanal_SPY,ret_semanal_TIP,ret_semanal_TLT,ret_semanal_USMV,ret_semanal_USO,ret_semanal_VEA,ret_semanal_VIG,ret_semanal_VLUE,ret_semanal_VNQ,ret_semanal_VOO,ret_semanal_VTI,ret_semanal_VTV,ret_semanal_VUG,ret_semanal_VWO,ret_semanal_XLB,ret_semanal_XLE,ret_semanal_XLF,ret_semanal_XLI,ret_semanal_XLK,ret_semanal_XLP,ret_semanal_XLRE,ret_semanal_XLU,ret_semanal_XLV,ret_semanal_XLY,volumen_log_diff_AGG,volumen_log_diff_BND,volumen_log_diff_DBC,volumen_log_diff_DIA,volumen_log_diff_DVY,volumen_log_diff_EEM,volumen_log_diff_EFA,volumen_log_diff_EWG,volumen_log_diff_EWJ,volumen_log_diff_EWQ,volumen_log_diff_EWT,volumen_log_diff_EWU,volumen_log_diff_EWZ,volumen_log_diff_FXI,volumen_log_diff_HYG,volumen_log_diff_IEF,volumen_log_diff_IEFA,volumen_log_diff_IEMG,volumen_log_diff_IJR,volumen_log_diff_INDA,volumen_log_diff_ITOT,volumen_log_diff_IVV,volumen_log_diff_IWD,volumen_log_diff_IWF,volumen_log_diff_IWM,volumen_log_diff_JNK,volumen_log_diff_LQD,volumen_log_diff_MCHI,volumen_log_diff_MDY,volumen_log_diff_MTUM,volumen_log_diff_QQQ,volumen_log_diff_QUAL,volumen_log_diff_SCHD,volumen_log_diff_SLV,volumen_log_diff_SPY,volumen_log_diff_TIP,volumen_log_diff_TLT,volumen_log_diff_USMV,volumen_log_diff_USO,volumen_log_diff_VEA,volumen_log_diff_VIG,volumen_log_diff_VLUE,volumen_log_diff_VNQ,volumen_log_diff_VOO,volumen_log_diff_VTI,volumen_log_diff_VTV,volumen_log_diff_VUG,volumen_log_diff_VWO,volumen_log_diff_XLB,volumen_log_diff_XLE,volumen_log_diff_XLF,volumen_log_diff_XLI,volumen_log_diff_XLK,volumen_log_diff_XLP,volumen_log_diff_XLRE,volumen_log_diff_XLU,volumen_log_diff_XLV,volumen_log_diff_XLY,target_AGG,target_BND,target_DBC,target_DIA,target_DVY,target_EEM,target_EFA,target_EWG,target_EWJ,target_EWQ,target_EWT,target_EWU,target_EWZ,target_FXI,target_HYG,target_IEF,target_IEFA,target_IEMG,target_IJR,target_INDA,target_ITOT,target_IVV,target_IWD,target_IWF,target_IWM,target_JNK,target_LQD,target_MCHI,target_MDY,target_MTUM,target_QQQ,target_QUAL,target_SCHD,target_SLV,target_SPY,target_TIP,target_TLT,target_USMV,target_USO,target_VEA,target_VIG,target_VLUE,target_VNQ,target_VOO,target_VTI,target_VTV,target_VUG,target_VWO,target_XLB,target_XLE,target_XLF,target_XLI,target_XLK,target_XLP,target_XLRE,target_XLU,target_XLV,target_XLY
0,2021-W01,2021,1,1,1,0,1,0,-0.002042,-0.002281,0.010383,0.007109,0.012052,0.012701,0.008017,0.005839,0.009299,0.006168,0.011315,0.012966,0.007608,0.010676,0.000716,-0.003652,0.008001,0.011714,0.017702,0.008355,0.009113,0.008249,0.010667,0.006692,0.017728,0.000804,-0.003056,0.012631,0.015672,0.012029,0.007735,0.006175,0.009732,-0.018838,0.008313,-0.001964,-0.010071,0.005358,0.022504,0.008591,0.006345,0.011569,0.002723,0.008261,0.009436,0.009851,0.007234,0.010313,0.016233,0.021808,0.015338,0.009091,0.005529,0.000674,0.001971,0.004901,0.009639,0.014278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000290,0.000183,0.000786,-0.001804,0.002915,-0.001320,-0.003544,-0.006671,-0.001652,-0.005888,0.003209,-0.004595,-0.004344,0.001238,-0.000596,0.000355,-0.003856,-0.001255,0.003731,-0.003339,-0.002004,-0.002917,-0.000227,-0.004766,0.002896,-0.000606,0.000559,0.000468,0.000526,-0.003981,-0.004546,-0.003442,0.000395,-

In [319]:
def crear_lags_wide(df, cols_base, lags):

    df = df.sort_values(["year", "semana"]).copy()

    for lag in range(1, lags + 1):
        lag_df = df[cols_base].shift(lag)
        lag_df.columns = [f"{col}_lag{lag}" for col in cols_base]
        df = pd.concat([df, lag_df], axis=1)

    return df

Dataset con 1 lag

In [320]:
cols_ret = [col for col in df_semanal.columns if col.startswith("ret_semanal_")]

weekly_semanal_1 = crear_lags_wide(
    df_semanal,
    cols_base=cols_ret,
    lags=1
)

pd.set_option("display.max_columns", None) 
weekly_semanal_1.head()

,week_key,year,semana,mes,inicio_mes,fin_mes,inicio_trimestre,fin_trimestre,ret_semanal_AGG,ret_semanal_BND,ret_semanal_DBC,ret_semanal_DIA,ret_semanal_DVY,ret_semanal_EEM,ret_semanal_EFA,ret_semanal_EWG,ret_semanal_EWJ,ret_semanal_EWQ,ret_semanal_EWT,ret_semanal_EWU,ret_semanal_EWZ,ret_semanal_FXI,ret_semanal_HYG,ret_semanal_IEF,ret_semanal_IEFA,ret_semanal_IEMG,ret_semanal_IJR,ret_semanal_INDA,ret_semanal_ITOT,ret_semanal_IVV,ret_semanal_IWD,ret_semanal_IWF,ret_semanal_IWM,ret_semanal_JNK,ret_semanal_LQD,ret_semanal_MCHI,ret_semanal_MDY,ret_semanal_MTUM,ret_semanal_QQQ,ret_semanal_QUAL,ret_semanal_SCHD,ret_semanal_SLV,ret_semanal_SPY,ret_semanal_TIP,ret_semanal_TLT,ret_semanal_USMV,ret_semanal_USO,ret_semanal_VEA,ret_semanal_VIG,ret_semanal_VLUE,ret_semanal_VNQ,ret_semanal_VOO,ret_semanal_VTI,ret_semanal_VTV,ret_semanal_VUG,ret_semanal_VWO,ret_semanal_XLB,ret_semanal_XLE,ret_semanal_XLF,ret_semanal_XLI,ret_semanal_XLK,ret_semanal_XLP,ret_semanal_XLRE,ret_semanal_XLU,ret_semanal_XLV,ret_semanal_XLY,volumen_log_diff_AGG,volumen_log_diff_BND,volumen_log_diff_DBC,volumen_log_diff_DIA,volumen_log_diff_DVY,volumen_log_diff_EEM,volumen_log_diff_EFA,volumen_log_diff_EWG,volumen_log_diff_EWJ,volumen_log_diff_EWQ,volumen_log_diff_EWT,volumen_log_diff_EWU,volumen_log_diff_EWZ,volumen_log_diff_FXI,volumen_log_diff_HYG,volumen_log_diff_IEF,volumen_log_diff_IEFA,volumen_log_diff_IEMG,volumen_log_diff_IJR,volumen_log_diff_INDA,volumen_log_diff_ITOT,volumen_log_diff_IVV,volumen_log_diff_IWD,volumen_log_diff_IWF,volumen_log_diff_IWM,volumen_log_diff_JNK,volumen_log_diff_LQD,volumen_log_diff_MCHI,volumen_log_diff_MDY,volumen_log_diff_MTUM,volumen_log_diff_QQQ,volumen_log_diff_QUAL,volumen_log_diff_SCHD,volumen_log_diff_SLV,volumen_log_diff_SPY,volumen_log_diff_TIP,volumen_log_diff_TLT,volumen_log_diff_USMV,volumen_log_diff_USO,volumen_log_diff_VEA,volumen_log_diff_VIG,volumen_log_diff_VLUE,volumen_log_diff_VNQ,volumen_log_diff_VOO,volumen_log_diff_VTI,volumen_log_diff_VTV,volumen_log_diff_VUG,volumen_log_diff_VWO,volumen_log_diff_XLB,volumen_log_diff_XLE,volumen_log_diff_XLF,volumen_log_diff_XLI,volumen_log_diff_XLK,volumen_log_diff_XLP,volumen_log_diff_XLRE,volumen_log_diff_XLU,volumen_log_diff_XLV,volumen_log_diff_XLY,target_AGG,target_BND,target_DBC,target_DIA,target_DVY,target_EEM,target_EFA,target_EWG,target_EWJ,target_EWQ,target_EWT,target_EWU,target_EWZ,target_FXI,target_HYG,target_IEF,target_IEFA,target_IEMG,target_IJR,target_INDA,target_ITOT,target_IVV,target_IWD,target_IWF,target_IWM,target_JNK,target_LQD,target_MCHI,target_MDY,target_MTUM,target_QQQ,target_QUAL,target_SCHD,target_SLV,target_SPY,target_TIP,target_TLT,target_USMV,target_USO,target_VEA,target_VIG,target_VLUE,target_VNQ,target_VOO,target_VTI,target_VTV,target_VUG,target_VWO,target_XLB,target_XLE,target_XLF,target_XLI,target_XLK,target_XLP,target_XLRE,target_XLU,target_XLV,target_XLY,ret_semanal_AGG_lag1,ret_semanal_BND_lag1,ret_semanal_DBC_lag1,ret_semanal_DIA_lag1,ret_semanal_DVY_lag1,ret_semanal_EEM_lag1,ret_semanal_EFA_lag1,ret_semanal_EWG_lag1,ret_semanal_EWJ_lag1,ret_semanal_EWQ_lag1,ret_semanal_EWT_lag1,ret_semanal_EWU_lag1,ret_semanal_EWZ_lag1,ret_semanal_FXI_lag1,ret_semanal_HYG_lag1,ret_semanal_IEF_lag1,ret_semanal_IEFA_lag1,ret_semanal_IEMG_lag1,ret_semanal_IJR_lag1,ret_semanal_INDA_lag1,ret_semanal_ITOT_lag1,ret_semanal_IVV_lag1,ret_semanal_IWD_lag1,ret_semanal_IWF_lag1,ret_semanal_IWM_lag1,ret_semanal_JNK_lag1,ret_semanal_LQD_lag1,ret_semanal_MCHI_lag1,ret_semanal_MDY_lag1,ret_semanal_MTUM_lag1,ret_semanal_QQQ_lag1,ret_semanal_QUAL_lag1,ret_semanal_SCHD_lag1,ret_semanal_SLV_lag1,ret_semanal_SPY_lag1,ret_semanal_TIP_lag1,ret_semanal_TLT_lag1,ret_semanal_USMV_lag1,ret_semanal_USO_lag1,ret_semanal_VEA_lag1,ret_semanal_VIG_lag1,ret_semanal_VLUE_lag1,ret_semanal_VNQ_lag1,ret_semanal_VOO_lag1,ret_semanal_VTI_lag1,ret_semanal_VTV_lag1,ret_semanal_VUG_lag1,ret_semanal_VWO_lag1,ret_semanal_XLB_lag1,ret_semanal_XLE_lag1,ret_semanal_XLF_lag1,ret_semanal_XLI_lag1,ret_s

Dataset con 2 lags

In [321]:
weekly_semanal_2 =crear_lags_wide(
    df_semanal,
    cols_base=cols_ret,
    lags=2
)


pd.set_option("display.max_columns", None) 
weekly_semanal_2.head()

,week_key,year,semana,mes,inicio_mes,fin_mes,inicio_trimestre,fin_trimestre,ret_semanal_AGG,ret_semanal_BND,ret_semanal_DBC,ret_semanal_DIA,ret_semanal_DVY,ret_semanal_EEM,ret_semanal_EFA,ret_semanal_EWG,ret_semanal_EWJ,ret_semanal_EWQ,ret_semanal_EWT,ret_semanal_EWU,ret_semanal_EWZ,ret_semanal_FXI,ret_semanal_HYG,ret_semanal_IEF,ret_semanal_IEFA,ret_semanal_IEMG,ret_semanal_IJR,ret_semanal_INDA,ret_semanal_ITOT,ret_semanal_IVV,ret_semanal_IWD,ret_semanal_IWF,ret_semanal_IWM,ret_semanal_JNK,ret_semanal_LQD,ret_semanal_MCHI,ret_semanal_MDY,ret_semanal_MTUM,ret_semanal_QQQ,ret_semanal_QUAL,ret_semanal_SCHD,ret_semanal_SLV,ret_semanal_SPY,ret_semanal_TIP,ret_semanal_TLT,ret_semanal_USMV,ret_semanal_USO,ret_semanal_VEA,ret_semanal_VIG,ret_semanal_VLUE,ret_semanal_VNQ,ret_semanal_VOO,ret_semanal_VTI,ret_semanal_VTV,ret_semanal_VUG,ret_semanal_VWO,ret_semanal_XLB,ret_semanal_XLE,ret_semanal_XLF,ret_semanal_XLI,ret_semanal_XLK,ret_semanal_XLP,ret_semanal_XLRE,ret_semanal_XLU,ret_semanal_XLV,ret_semanal_XLY,volumen_log_diff_AGG,volumen_log_diff_BND,volumen_log_diff_DBC,volumen_log_diff_DIA,volumen_log_diff_DVY,volumen_log_diff_EEM,volumen_log_diff_EFA,volumen_log_diff_EWG,volumen_log_diff_EWJ,volumen_log_diff_EWQ,volumen_log_diff_EWT,volumen_log_diff_EWU,volumen_log_diff_EWZ,volumen_log_diff_FXI,volumen_log_diff_HYG,volumen_log_diff_IEF,volumen_log_diff_IEFA,volumen_log_diff_IEMG,volumen_log_diff_IJR,volumen_log_diff_INDA,volumen_log_diff_ITOT,volumen_log_diff_IVV,volumen_log_diff_IWD,volumen_log_diff_IWF,volumen_log_diff_IWM,volumen_log_diff_JNK,volumen_log_diff_LQD,volumen_log_diff_MCHI,volumen_log_diff_MDY,volumen_log_diff_MTUM,volumen_log_diff_QQQ,volumen_log_diff_QUAL,volumen_log_diff_SCHD,volumen_log_diff_SLV,volumen_log_diff_SPY,volumen_log_diff_TIP,volumen_log_diff_TLT,volumen_log_diff_USMV,volumen_log_diff_USO,volumen_log_diff_VEA,volumen_log_diff_VIG,volumen_log_diff_VLUE,volumen_log_diff_VNQ,volumen_log_diff_VOO,volumen_log_diff_VTI,volumen_log_diff_VTV,volumen_log_diff_VUG,volumen_log_diff_VWO,volumen_log_diff_XLB,volumen_log_diff_XLE,volumen_log_diff_XLF,volumen_log_diff_XLI,volumen_log_diff_XLK,volumen_log_diff_XLP,volumen_log_diff_XLRE,volumen_log_diff_XLU,volumen_log_diff_XLV,volumen_log_diff_XLY,target_AGG,target_BND,target_DBC,target_DIA,target_DVY,target_EEM,target_EFA,target_EWG,target_EWJ,target_EWQ,target_EWT,target_EWU,target_EWZ,target_FXI,target_HYG,target_IEF,target_IEFA,target_IEMG,target_IJR,target_INDA,target_ITOT,target_IVV,target_IWD,target_IWF,target_IWM,target_JNK,target_LQD,target_MCHI,target_MDY,target_MTUM,target_QQQ,target_QUAL,target_SCHD,target_SLV,target_SPY,target_TIP,target_TLT,target_USMV,target_USO,target_VEA,target_VIG,target_VLUE,target_VNQ,target_VOO,target_VTI,target_VTV,target_VUG,target_VWO,target_XLB,target_XLE,target_XLF,target_XLI,target_XLK,target_XLP,target_XLRE,target_XLU,target_XLV,target_XLY,ret_semanal_AGG_lag1,ret_semanal_BND_lag1,ret_semanal_DBC_lag1,ret_semanal_DIA_lag1,ret_semanal_DVY_lag1,ret_semanal_EEM_lag1,ret_semanal_EFA_lag1,ret_semanal_EWG_lag1,ret_semanal_EWJ_lag1,ret_semanal_EWQ_lag1,ret_semanal_EWT_lag1,ret_semanal_EWU_lag1,ret_semanal_EWZ_lag1,ret_semanal_FXI_lag1,ret_semanal_HYG_lag1,ret_semanal_IEF_lag1,ret_semanal_IEFA_lag1,ret_semanal_IEMG_lag1,ret_semanal_IJR_lag1,ret_semanal_INDA_lag1,ret_semanal_ITOT_lag1,ret_semanal_IVV_lag1,ret_semanal_IWD_lag1,ret_semanal_IWF_lag1,ret_semanal_IWM_lag1,ret_semanal_JNK_lag1,ret_semanal_LQD_lag1,ret_semanal_MCHI_lag1,ret_semanal_MDY_lag1,ret_semanal_MTUM_lag1,ret_semanal_QQQ_lag1,ret_semanal_QUAL_lag1,ret_semanal_SCHD_lag1,ret_semanal_SLV_lag1,ret_semanal_SPY_lag1,ret_semanal_TIP_lag1,ret_semanal_TLT_lag1,ret_semanal_USMV_lag1,ret_semanal_USO_lag1,ret_semanal_VEA_lag1,ret_semanal_VIG_lag1,ret_semanal_VLUE_lag1,ret_semanal_VNQ_lag1,ret_semanal_VOO_lag1,ret_semanal_VTI_lag1,ret_semanal_VTV_lag1,ret_semanal_VUG_lag1,ret_semanal_VWO_lag1,ret_semanal_XLB_lag1,ret_semanal_XLE_lag1,ret_semanal_XLF_lag1,ret_semanal_XLI_lag1,ret_s

In [322]:
weekly_semanal_3 = crear_lags_wide(
    df_semanal,
    cols_base=cols_ret,
    lags=3
)


In [323]:
weekly_semanal_4 = crear_lags_wide(
    df_semanal,
    cols_base=cols_ret,
    lags=4
)


In [324]:
import os

def guardar_dataset(df, nombre_archivo):
    
    out_dir = "../Datos_csv"
    os.makedirs(out_dir, exist_ok=True)

    path = os.path.join(out_dir, nombre_archivo)

    df.to_csv(path, index=False)

    print("Guardado:")
    print(" -", path, df.shape)

In [325]:
df_semanal = df_semanal.dropna().reset_index(drop=True)
weekly_semanal_1 = weekly_semanal_1.dropna().reset_index(drop=True)
weekly_semanal_2 = weekly_semanal_2.dropna().reset_index(drop=True)
weekly_semanal_3 = weekly_semanal_3.dropna().reset_index(drop=True)
weekly_semanal_4 = weekly_semanal_4.dropna().reset_index(drop=True)



In [326]:
# # columnas a eliminar como features
# cols_borrar = ["week_key", "year", "semana", "mes"]

# # eliminar solo si existen
# df_semanal = df_semanal.drop(columns=cols_borrar, errors="ignore")
# weekly_semanal_1 = weekly_semanal_1.drop(columns=cols_borrar, errors="ignore")
# weekly_semanal_2 = weekly_semanal_2.drop(columns=cols_borrar, errors="ignore")
# weekly_semanal_3 = weekly_semanal_3.drop(columns=cols_borrar, errors="ignore")
# weekly_semanal_4 = weekly_semanal_4.drop(columns=cols_borrar, errors="ignore")


In [327]:
pd.set_option("display.max_columns", None) 
df_semanal.head()

,week_key,year,semana,mes,inicio_mes,fin_mes,inicio_trimestre,fin_trimestre,ret_semanal_AGG,ret_semanal_BND,ret_semanal_DBC,ret_semanal_DIA,ret_semanal_DVY,ret_semanal_EEM,ret_semanal_EFA,ret_semanal_EWG,ret_semanal_EWJ,ret_semanal_EWQ,ret_semanal_EWT,ret_semanal_EWU,ret_semanal_EWZ,ret_semanal_FXI,ret_semanal_HYG,ret_semanal_IEF,ret_semanal_IEFA,ret_semanal_IEMG,ret_semanal_IJR,ret_semanal_INDA,ret_semanal_ITOT,ret_semanal_IVV,ret_semanal_IWD,ret_semanal_IWF,ret_semanal_IWM,ret_semanal_JNK,ret_semanal_LQD,ret_semanal_MCHI,ret_semanal_MDY,ret_semanal_MTUM,ret_semanal_QQQ,ret_semanal_QUAL,ret_semanal_SCHD,ret_semanal_SLV,ret_semanal_SPY,ret_semanal_TIP,ret_semanal_TLT,ret_semanal_USMV,ret_semanal_USO,ret_semanal_VEA,ret_semanal_VIG,ret_semanal_VLUE,ret_semanal_VNQ,ret_semanal_VOO,ret_semanal_VTI,ret_semanal_VTV,ret_semanal_VUG,ret_semanal_VWO,ret_semanal_XLB,ret_semanal_XLE,ret_semanal_XLF,ret_semanal_XLI,ret_semanal_XLK,ret_semanal_XLP,ret_semanal_XLRE,ret_semanal_XLU,ret_semanal_XLV,ret_semanal_XLY,volumen_log_diff_AGG,volumen_log_diff_BND,volumen_log_diff_DBC,volumen_log_diff_DIA,volumen_log_diff_DVY,volumen_log_diff_EEM,volumen_log_diff_EFA,volumen_log_diff_EWG,volumen_log_diff_EWJ,volumen_log_diff_EWQ,volumen_log_diff_EWT,volumen_log_diff_EWU,volumen_log_diff_EWZ,volumen_log_diff_FXI,volumen_log_diff_HYG,volumen_log_diff_IEF,volumen_log_diff_IEFA,volumen_log_diff_IEMG,volumen_log_diff_IJR,volumen_log_diff_INDA,volumen_log_diff_ITOT,volumen_log_diff_IVV,volumen_log_diff_IWD,volumen_log_diff_IWF,volumen_log_diff_IWM,volumen_log_diff_JNK,volumen_log_diff_LQD,volumen_log_diff_MCHI,volumen_log_diff_MDY,volumen_log_diff_MTUM,volumen_log_diff_QQQ,volumen_log_diff_QUAL,volumen_log_diff_SCHD,volumen_log_diff_SLV,volumen_log_diff_SPY,volumen_log_diff_TIP,volumen_log_diff_TLT,volumen_log_diff_USMV,volumen_log_diff_USO,volumen_log_diff_VEA,volumen_log_diff_VIG,volumen_log_diff_VLUE,volumen_log_diff_VNQ,volumen_log_diff_VOO,volumen_log_diff_VTI,volumen_log_diff_VTV,volumen_log_diff_VUG,volumen_log_diff_VWO,volumen_log_diff_XLB,volumen_log_diff_XLE,volumen_log_diff_XLF,volumen_log_diff_XLI,volumen_log_diff_XLK,volumen_log_diff_XLP,volumen_log_diff_XLRE,volumen_log_diff_XLU,volumen_log_diff_XLV,volumen_log_diff_XLY,target_AGG,target_BND,target_DBC,target_DIA,target_DVY,target_EEM,target_EFA,target_EWG,target_EWJ,target_EWQ,target_EWT,target_EWU,target_EWZ,target_FXI,target_HYG,target_IEF,target_IEFA,target_IEMG,target_IJR,target_INDA,target_ITOT,target_IVV,target_IWD,target_IWF,target_IWM,target_JNK,target_LQD,target_MCHI,target_MDY,target_MTUM,target_QQQ,target_QUAL,target_SCHD,target_SLV,target_SPY,target_TIP,target_TLT,target_USMV,target_USO,target_VEA,target_VIG,target_VLUE,target_VNQ,target_VOO,target_VTI,target_VTV,target_VUG,target_VWO,target_XLB,target_XLE,target_XLF,target_XLI,target_XLK,target_XLP,target_XLRE,target_XLU,target_XLV,target_XLY
0,2021-W02,2021,2,1,0,0,0,0,0.000290,0.000183,0.000786,-0.001804,0.002915,-0.001320,-0.003544,-0.006671,-0.001652,-0.005888,0.003209,-0.004595,-0.004344,0.001238,-0.000596,0.000355,-0.003856,-0.001255,0.003731,-0.003339,-0.002004,-0.002917,-0.000227,-0.004766,0.002896,-0.000606,0.000559,0.000468,0.000526,-0.003981,-0.004546,-0.003442,0.000395,-0.004822,-0.002938,0.000599,0.000660,-0.003214,-0.000509,-0.003780,-0.002962,0.005531,0.003674,-0.002903,-0.002210,0.000278,-0.005427,0.000152,-0.003082,0.006321,0.000129,-0.001748,-0.005175,-0.003833,0.003778,0.002107,-0.000717,-0.003575,-0.117361,-0.086574,-0.539732,-0.459750,-0.292106,-0.256894,-0.017097,-0.504441,-0.288985,0.146919,-0.384660,-0.024032,-0.021922,-0.680665,0.200763,-0.129573,0.133340,0.202425,0.300938,0.028507,-0.114187,0.205928,0.176201,0.052224,-0.149952,-0.075401,0.009488,-0.179850,-0.440872,0.933530,-0.243510,0.630144,-0.176504,-0.481412,-0.251597,-0.198469,-0.256328,0.642472,-0.516593,-0.123466,-0.090498,0.430339,0.262899,-0.227624,-0.393088,0.233110,0.333712,-0.265088,-0.673632,-0.004085,-0.095183,-0.250324,-0.153681,-0.279392

In [328]:
df_semanal.shape

(264, 182)

In [329]:
weekly_semanal_2.shape

(263, 298)

In [330]:
guardar_dataset(df_semanal, "df_semanal.csv")
guardar_dataset(weekly_semanal_1, "df_semanal_1.csv")
guardar_dataset(weekly_semanal_2, "df_semanal_2.csv")
guardar_dataset(weekly_semanal_3, "df_semanal_3.csv")
guardar_dataset(weekly_semanal_4, "df_semanal_4.csv")


Guardado:
 - ../Datos_csv\df_semanal.csv (264, 182)


Guardado:
 - ../Datos_csv\df_semanal_1.csv (264, 240)
Guardado:
 - ../Datos_csv\df_semanal_2.csv (263, 298)
Guardado:
 - ../Datos_csv\df_semanal_3.csv (262, 356)
Guardado:
 - ../Datos_csv\df_semanal_4.csv (261, 414)
